# web-researcher-mcp Python SDK — Examples

**What this notebook covers**

| # | Scenario | Why it matters |
|---|----------|----------------|
| 1 | Web search with result verification | Catches hallucinated or dead links before they reach your users |
| 2 | Verify a citation (DOI / URL) | AI citations are wrong ~60% of the time; this flags fabricated papers |
| 3 | Audit a whole bibliography | One call checks every reference in a paper or report for retractions and broken links |
| 4 | Academic literature search | PubMed + OpenAlex, returning open-access PDFs and DOIs |
| 5 | News monitoring | Structured headlines + timestamps, no scraping required |
| 6 | Deep research (sequential search) | Multi-step research that persists state across LLM context windows |
| 7 | Economic data | World Bank / OECD time-series without wrestling with their SDMXs APIs |

**Zero-config providers** — cells 1, 2, 3, 5 use DuckDuckGo + CrossRef, which need no API key.  
Set `GOOGLE_CUSTOM_SEARCH_API_KEY` + `GOOGLE_CUSTOM_SEARCH_ID` in the env block below to upgrade to Google.

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zoharbabin/web-researcher-mcp/blob/main/examples/web_researcher_sdk_examples.ipynb)

## Install

In [ ]:
# Install the SDK. The wheel bundles the signed Go binary — no Go toolchain needed.
# On Google Colab this installs the Linux/amd64 binary automatically.
%pip install -q web-researcher-mcp

In [ ]:
import asyncio
import os
from web_researcher_mcp import WebResearcherClient

# Optional: set API keys here or via environment variables.
# Leave blank to use the free DuckDuckGo fallback.
# os.environ["GOOGLE_CUSTOM_SEARCH_API_KEY"] = "your-key"
# os.environ["GOOGLE_CUSTOM_SEARCH_ID"] = "your-cx-id"

# Helper: notebooks can't use `async with` at the top level in older Jupyter.
# Use WebResearcherClient.sync() for a synchronous wrapper instead.
client = WebResearcherClient.sync()
client.__enter__()
print("Server started. SDK version:", client._async_client._server._proc is not None and "ok" or "connected")

---
## 1 — Web search with source verification

**Why**: AI-generated summaries often cite pages that don't exist or have moved.
Calling `verify_citation` on every result URL catches dead links *before* they
end up in a report or prompt.

This pattern — search → verify — is the simplest anti-hallucination loop.

In [ ]:
query = "CRISPR off-target effects clinical trials 2024"

results = client.web_search(query, num_results=5)

print(f"Query: {results.query}")
print(f"Results: {results.resultCount}\n")

for r in results.results:
    check = client.verify_citation(r.url)
    status = "✓ live" if check.httpStatus in (200, 301, 302) else "✗ dead"
    retracted = " [RETRACTED]" if (check.retractionStatus and check.retractionStatus.retracted) else ""
    print(f"{status}{retracted}  {r.title}")
    print(f"   {r.url}")
    if r.snippet:
        print(f"   {r.snippet[:120]}…")
    print()

---
## 2 — Verify a single citation

**Why**: Studies show AI models fabricate ~60% of academic citations (Stanford 2023,
CJR 2024). `verify_citation` checks CrossRef for existence, retraction status
from Crossref Retraction Watch, and whether the URL still resolves.

Pass a DOI, a URL, or a free-text title — the tool handles all three.

In [ ]:
# A real, well-known paper:
real_doi = "10.1038/nature14538"  # Doudna & Charpentier CRISPR Nature 2015

# A fabricated DOI (does not exist):
fake_doi = "10.9999/definitely-not-a-real-paper-xyz-12345"

for label, citation in [("Real DOI", real_doi), ("Fake DOI", fake_doi)]:
    result = client.verify_citation(citation)
    print(f"--- {label}: {citation} ---")
    print(f"  exists:    {result.exists}")
    print(f"  retracted: {result.retractionStatus.retracted if result.retractionStatus else None}")
    print(f"  link live: {result.httpStatus in (200, 301, 302)}")
    if result.matchedRecord:
        rec = result.matchedRecord  # raw Crossref/OpenAlex record (dict)
        print(f"  title:     {rec.get('title') if isinstance(rec, dict) else rec}")
        print(f"  authors:   {rec.get('author') if isinstance(rec, dict) else None}")
    print()

---
## 3 — Audit a whole bibliography

**Why**: A bibliography in a paper, report, or AI response can contain dozens of
references. Checking each one by hand is tedious and error-prone. `audit_bibliography`
checks all of them in one call: existence, retraction, dead links, and (optionally)
whether each source actually supports the claim it's cited for.

Paste BibTeX, RIS, or CSL-JSON, or pass a list of `{title, doi, url}` dicts.

In [ ]:
bibliography_entries = [
    # Real paper:
    {"doi": "10.1038/nature14538", "title": "A programmable dual-RNA-guided DNA endonuclease"},
    # Fabricated entry (should be flagged as not found):
    {"doi": "10.9999/fabricated-paper-xyz", "title": "Completely made up study on something"},
    # Real URL:
    {"url": "https://www.nature.com/articles/nature14538"},
]

audit = client.audit_bibliography(entries=bibliography_entries)

print("Bibliography audit summary")
print("=" * 40)
if audit.summary:
    s = audit.summary
    print(f"  total:      {s.total}")
    print(f"  not found:  {s.notFound}")
    print(f"  retracted:  {s.retracted}")
    print(f"  dead links: {s.deadLink}")

print()
for entry in (audit.entries or []):
    flag = "✓" if entry.exists else "✗ NOT FOUND"
    retracted = " [RETRACTED]" if (entry.retractionStatus or {}).get("retracted") else ""
    print(f"{flag}{retracted}  {entry.doi or entry.url or entry.title}")

---
## 4 — Academic literature search

**Why**: Google Scholar doesn't have a public API; PubMed and OpenAlex do.
`academic_search` wraps both, returns DOIs and open-access PDF links, and
normalises the messy `authors` / `year` / `journal` fields across providers.

PubMed is keyless at ~100 req/day; pass `SEMANTIC_SCHOLAR_API_KEY` for higher limits.

In [ ]:
papers = client.academic_search(
    "large language model hallucination detection",
    num_results=5,
    open_access=True,   # only return papers with a free PDF
    source="pubmed",
)

print(f"Found {papers.totalResults} papers (showing {len(papers.papers)})\n")

for p in papers.papers:
    print(f"  {p.title}")
    print(f"  Authors: {', '.join(p.authors[:3]) if p.authors else 'unknown'}"
          + (" et al." if p.authors and len(p.authors) > 3 else ""))
    print(f"  Year: {p.year}  |  DOI: {p.doi or 'n/a'}")
    if p.pdfUrl:
        print(f"  PDF: {p.pdfUrl}")
    print()

---
## 5 — News monitoring

**Why**: News APIs typically return snippets, not full article text, and they
charge per-request. `news_search` uses DuckDuckGo (free, keyless) as the default
and returns structured headlines, sources, and publish timestamps.

Pass `provider="brave"` with a `BRAVE_API_KEY` for real-time results.

In [ ]:
news = client.news_search(
    "artificial intelligence regulation",
    num_results=5,
    time_range="week",  # "day" | "week" | "month" | "year"
)

print(f"Recent news: {len(news.articles)} articles\n")

for article in news.articles:
    print(f"  [{article.publishedAt or 'n/a'}] {article.title}")
    print(f"   Source: {article.source}")
    print(f"   URL:    {article.url}")
    if article.snippet:
        print(f"   {article.snippet[:120]}…")
    print()

---
## 6 — Deep research with sequential search

**Why**: A single `web_search` call can't follow up on what it finds.
`sequential_search` is a stateful research engine: each step can branch,
revise, or extend previous steps. The session ID lets you resume after an LLM
context window expires.

This is the tool to use when you want an AI to conduct *genuine* multi-step
research rather than a single lookup.

In [ ]:
# Step 1: start the research session
step1 = client.sequential_search(
    search_step="Search for recent breakthroughs in mRNA vaccine technology beyond COVID-19",
    step_number=1,
    next_step_needed=True,
    researchGoal="Understand current mRNA vaccine pipeline and future applications",
    depth="comprehensive",
)

session_id = step1.sessionId
print(f"Session: {session_id}")
print(f"Step 1 complete. Sources found: {len(step1.sources or [])}")
for src in (step1.sources or [])[:3]:
    print(f"  - {getattr(src, 'title', src)}")

print()

In [ ]:
# Step 2: follow up on a specific angle identified in step 1
step2 = client.sequential_search(
    search_step="Find clinical trial results for mRNA cancer vaccines specifically",
    step_number=2,
    next_step_needed=False,  # last step — export after this
    sessionId=session_id,
    depth="comprehensive",
)

print(f"Step 2 complete. Total session sources: {len(step2.sources or [])}")
print()

# Export the research as a Markdown report
report = client.research_export(sessionId=session_id, format="markdown", verify_links=False)
print("=== Research Report (first 800 chars) ===")
content = report.document or ""
print(content[:800])
if len(content) > 800:
    print("... (truncated)")

---
## 7 — Economic data (World Bank / OECD)

**Why**: The World Bank and OECD offer rich open data but their APIs are verbose
and poorly documented. `econ_search` wraps both behind a single natural-language
query interface, normalises units, and returns structured time-series.

All keyless — no API registration required.

In [ ]:
data = client.econ_search(
    query="GDP per capita",
    country="US",
    date_from="2015",
    date_to="2024",
    provider="worldbank",
)

# Top-level response fields: query, provider, country, seriesId, resultCount
print(f"Series ID: {data.seriesId or 'GDP per capita'}")
print(f"Provider:  {data.provider or 'worldbank'}")
print(f"Country:   {data.country or 'US'}")
print()

# Each result has: date, value, units, source, title
for point in (data.results or []):
    year  = point.date or '?'
    value = point.value
    units = point.units or ''
    if value is not None:
        print(f"  {year}: {value:,.0f} {units}" if isinstance(value, (int, float)) else f"  {year}: {value}")

---
## Clean up

In [ ]:
# Stop the background Go binary gracefully.
client.__exit__(None, None, None)
print("Server stopped.")

---
## Async usage (for Jupyter 7+ and IPython)

Newer Jupyter kernels have a running event loop, so you can use `async with` directly:

In [ ]:
# This cell only works in Jupyter 7+ / IPython with an active event loop.
# In older Jupyter, use WebResearcherClient.sync() as shown above.

async def demo_async():
    async with WebResearcherClient() as c:
        result = await c.web_search("Python asyncio best practices", num_results=3)
        for r in result.results:
            print(f"  {r.title}")
            print(f"  {r.url}")

# Uncomment one of these depending on your environment:
# await demo_async()             # Jupyter 7+ / IPython
# asyncio.run(demo_async())      # standalone script
print("(Uncomment the line above to run.)")